In [1]:
import os
import fastf1
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score)

os.makedirs('cache_dir', exist_ok=True)

fastf1.Cache.enable_cache('cache_dir')

In [2]:
qualifying21 = fastf1.get_session(2021, 'Belgium', 'Q')
qualifying22 = fastf1.get_session(2022, 'Belgium', 'Q')
qualifying23 = fastf1.get_session(2023, 'Belgium', 'Q')
qualifying24 = fastf1.get_session(2024, 'Belgium', 'Q')
qualifying25 = fastf1.get_session(2025, 'Belgium', 'Q')
qualifying26 = fastf1.get_session(2026, 'Belgium', 'Q')

race21 = fastf1.get_session(2021, 'Belgium', 'R')
race22 = fastf1.get_session(2022, 'Belgium', 'R')
race23 = fastf1.get_session(2023, 'Belgium', 'R')
race24 = fastf1.get_session(2024, 'Belgium', 'R')
race25 = fastf1.get_session(2025, 'Belgium', 'R')
race26 = fastf1.get_session(2026, 'Belgium', 'R')

In [3]:
qualifying21.load(laps=True, telemetry=True, weather=True)
qualifying22.load(laps=True, telemetry=True, weather=True)
qualifying23.load(laps=True, telemetry=True, weather=True)
qualifying24.load(laps=True, telemetry=True, weather=True)
qualifying25.load(laps=True, telemetry=True, weather=True)
qualifying26.load(laps=True, telemetry=True, weather=True)

race21.load(laps=True, telemetry=True, weather=True)
race22.load(laps=True, telemetry=True, weather=True)
race23.load(laps=True, telemetry=True, weather=True)
race24.load(laps=True, telemetry=True, weather=True)
race25.load(laps=True, telemetry=True, weather=True)
race26.load(laps=True, telemetry=True, weather=True)

core           INFO 	Loading data for Belgian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['33', '63', '44', '3', '5', '10', '11', '77', '31', '4', '16', '6', '55', '14', '18', '99', '22', '47', '7', '9']
core           INFO 	Loading data for Belgian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cache

In [4]:
sessions = [qualifying21, qualifying22, qualifying23, qualifying24, qualifying25, qualifying26]
races = [race21, race22, race23, race24, race25]   # only 5 — no race26 yet
years = [2021, 2022, 2023, 2024, 2025, 2026]
r_years = [2021, 2022, 2023, 2024, 2025]

quali_list = []
for session, year in zip(sessions, years):
    results = session.results.copy()
    results['Year'] = year
    quali_list.append(results)

race_list = []
for race, year in zip(races, r_years):
    race_results = race.results.copy()
    race_results['Year'] = year
    race_list.append(race_results)

quali_df = pd.concat(quali_list, ignore_index=True)
race_df = pd.concat(race_list, ignore_index=True)

In [5]:
quali_df.head(20)

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Year
0,33,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,0600EF,red_bull,Max,Verstappen,Max Verstappen,...,,NaN,0 days 00:01:58.717000,0 days 00:01:56.559000,0 days 00:01:59.765000,NaT,,NaN,NaN,2021
1,63,G RUSSELL,RUS,russell,Williams,005AFF,williams,George,Russell,George Russell,...,,NaN,0 days 00:01:59.864000,0 days 00:01:56.950000,0 days 00:02:00.086000,NaT,,NaN,NaN,2021
2,44,L HAMILTON,HAM,hamilton,Mercedes,00D2BE,mercedes,Lewis,Hamilton,Lewis Hamilton,...,,NaN,0 days 00:01:59.218000,0 days 00:01:56.229000,0 days 00:02:00.099000,NaT,,NaN,NaN,2021
3,3,D RICCIARDO,RIC,ricciardo,McLaren,FF9800,mclaren,Daniel,Ricciardo,Daniel Ricciardo,...,,NaN,0 days 00:02:01.583000,0 days 00:01:57.127000,0 days 00:02:00.864000,NaT,,NaN,NaN,2021
4,5,S VETTEL,VET,vettel,Aston Martin,006F62,aston_martin,Sebastian,Vettel,Sebastian Vettel,...,,NaN,0 days 00:02:00.175000,0 days 00:01:56.814000,0 days 00:02:00.935000,NaT,,NaN,NaN,2021
5,10,P GASLY,GAS,gasly,AlphaTauri,2B4562,alphatauri,Pierre,Gasly,Pierre Gasly,...,,NaN,0 days 00:02:00.387000,0 days 00:01:56.440000,0 days 00:02:01.164000,NaT,,NaN,NaN,2021
6,11,S PEREZ,PER,perez,Red Bull Racing,0600EF,red_bull,Sergio,Perez,Sergio Perez,...,,NaN,0 days 00:01:59.334000,0 days 00:01:56.886000,0 days 00:02:02.112000,NaT,,NaN,NaN,2021
7,77,V BOTTAS,BOT,bottas,Mercedes,00D2BE,mercedes,Valtteri,Bottas,Valtteri Bottas,...,,NaN,0 days 00:01:59.870000,0 days 00:01:56.295000,0 days 00:02:02.502000,NaT,,NaN,NaN,2021
8,31,E OCON,OCO,ocon,Alpine,0090FF,alpine,Esteban,Ocon,Esteban Ocon,...,,NaN,0 days 00:02:01.824000,0 days 00:01:57.354000,0 days 00:02:03.513000,NaT,,NaN,NaN,2021
9,4,L NORRIS,NOR,norris,McLaren,FF9800,mclaren,Lando,Norris,Lando Norris,...,,NaN,0 days 00:01:58.301000,0 days 00:01:56.025000,NaT,NaT,,NaN,NaN,2021


In [6]:
quali_df.columns

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps',
       'Year'],
      dtype='object')

In [7]:
print(quali_df.dtypes)

DriverNumber                   object
BroadcastName                  object
Abbreviation                   object
DriverId                       object
TeamName                       object
TeamColor                      object
TeamId                         object
FirstName                      object
LastName                       object
FullName                       object
HeadshotUrl                    object
CountryCode                    object
Position                      float64
ClassifiedPosition             object
GridPosition                  float64
Q1                    timedelta64[ns]
Q2                    timedelta64[ns]
Q3                    timedelta64[ns]
Time                  timedelta64[ns]
Status                         object
Points                        float64
Laps                          float64
Year                            int64
dtype: object


In [8]:
print(quali_df.isna().sum())

DriverNumber            0
BroadcastName           0
Abbreviation            0
DriverId                0
TeamName                0
TeamColor               0
TeamId                  0
FirstName               0
LastName                0
FullName                0
HeadshotUrl             0
CountryCode             0
Position                0
ClassifiedPosition      0
GridPosition          122
Q1                      0
Q2                     31
Q3                     64
Time                  122
Status                  0
Points                122
Laps                  122
Year                    0
dtype: int64


In [9]:
quali_df = quali_df.drop(columns=['BroadcastName', 'DriverId', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'Points', 'ClassifiedPosition', 'CountryCode', 'Status', 'Laps', 'GridPosition', 'Time'], errors='ignore')
quali_df.head()

,DriverNumber,Abbreviation,TeamName,Position,Q1,Q2,Q3,Year
0,33,VER,Red Bull Racing,1.0,0 days 00:01:58.717000,0 days 00:01:56.559000,0 days 00:01:59.765000,2021
1,63,RUS,Williams,2.0,0 days 00:01:59.864000,0 days 00:01:56.950000,0 days 00:02:00.086000,2021
2,44,HAM,Mercedes,3.0,0 days 00:01:59.218000,0 days 00:01:56.229000,0 days 00:02:00.099000,2021
3,3,RIC,McLaren,4.0,0 days 00:02:01.583000,0 days 00:01:57.127000,0 days 00:02:00.864000,2021
4,5,VET,Aston Martin,5.0,0 days 00:02:00.175000,0 days 00:01:56.814000,0 days 00:02:00.935000,2021


### Pre-processing

In [10]:
# Q1 Q2 Q3 columns have 0 days 00:01:46.304000, but for the project it would be best to keep it as secondsfor col in ['Q1', 'Q2', 'Q3']
for col in ['Q1', 'Q2', 'Q3']:
    quali_df[col] = quali_df[col].dt.total_seconds().round(3)

In [11]:
print(quali_df[['Q1', 'Q2', 'Q3']].isna().sum())

Q1     0
Q2    31
Q3    64
dtype: int64


In [12]:
print(quali_df.dtypes)

DriverNumber     object
Abbreviation     object
TeamName         object
Position        float64
Q1              float64
Q2              float64
Q3              float64
Year              int64
dtype: object


In [13]:
quali_df.fillna(0, inplace=True)
print(quali_df.isna().sum())

DriverNumber    0
Abbreviation    0
TeamName        0
Position        0
Q1              0
Q2              0
Q3              0
Year            0
dtype: int64


In [14]:
quali_df.head(22)

,DriverNumber,Abbreviation,TeamName,Position,Q1,Q2,Q3,Year
0,33,VER,Red Bull Racing,1.0,118.717,116.559,119.765,2021
1,63,RUS,Williams,2.0,119.864,116.950,120.086,2021
2,44,HAM,Mercedes,3.0,119.218,116.229,120.099,2021
3,3,RIC,McLaren,4.0,121.583,117.127,120.864,2021
4,5,VET,Aston Martin,5.0,120.175,116.814,120.935,2021
5,10,GAS,AlphaTauri,6.0,120.387,116.440,121.164,2021
6,11,PER,Red Bull Racing,7.0,119.334,116.886,122.112,2021
7,77,BOT,Mercedes,8.0,119.870,116.295,122.502,2021
8,31,OCO,Alpine,9.0,121.824,117.354,123.513,2021
9,4,NOR,McLaren,10.0,118.301,116.025,0.000,2021


In [15]:
# Encoding team name
quali_df = pd.get_dummies(quali_df, columns=['TeamName'], drop_first=True)

### Feature engineering

In [16]:
# Best quali time
quali_df['BestQualiTime'] = quali_df[['Q1', 'Q2', 'Q3']].min(axis=1, skipna=True)

In [17]:
# Quali improvement gap
quali_df['Q2_Q1_Improvement'] = quali_df['Q1'] - quali_df['Q2']
quali_df['Q3_Q1_Improvement'] = quali_df['Q2'] - quali_df['Q3']

In [18]:
# driver_lookup = quali_df[['DriverNumber', 'Abbreviation']].copy()
# quali_df_model = quali_df.drop(columns=['DriverNumber', 'Abbreviation'])


In [19]:
quali_df.head()

,DriverNumber,Abbreviation,Position,Q1,Q2,Q3,Year,TeamName_Alfa Romeo Racing,TeamName_AlphaTauri,TeamName_Alpine,...,TeamName_Kick Sauber,TeamName_McLaren,TeamName_Mercedes,TeamName_RB,TeamName_Racing Bulls,TeamName_Red Bull Racing,TeamName_Williams,BestQualiTime,Q2_Q1_Improvement,Q3_Q1_Improvement
0,33,VER,1.0,118.717,116.559,119.765,2021,False,False,False,...,False,False,False,False,False,True,False,116.559,2.158,-3.206
1,63,RUS,2.0,119.864,116.950,120.086,2021,False,False,False,...,False,False,False,False,False,False,True,116.950,2.914,-3.136
2,44,HAM,3.0,119.218,116.229,120.099,2021,False,False,False,...,False,False,True,False,False,False,False,116.229,2.989,-3.870
3,3,RIC,4.0,121.583,117.127,120.864,2021,False,False,False,...,False,True,False,False,False,False,False,117.127,4.456,-3.737
4,5,VET,5.0,120.175,116.814,120.935,2021,False,False,False,...,False,False,False,False,False,False,False,116.814,3.361,-4.121


### Race results for target variable

In [20]:
race_df.columns

Index(['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName',
       'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName',
       'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition',
       'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps',
       'Year'],
      dtype='object')

In [21]:
race_df_model = race_df[['Year', 'Abbreviation', 'Position']].copy()
race_df_model = race_df_model.rename(columns={'Position': 'RacePosition'})
race_df_model.head()

,Year,Abbreviation,RacePosition
0,2021,VER,1.0
1,2021,RUS,2.0
2,2021,HAM,3.0
3,2021,RIC,4.0
4,2021,VET,5.0


In [22]:
race_df_model['RacePosition'].isna().sum()  # should be 0

np.int64(0)

In [23]:
merged_df = quali_df.merge(race_df_model, on=['Year', 'Abbreviation'], how='left')
merged_df['Winner'] = (merged_df['RacePosition'] == 1).astype(int)
merged_df.head()

,DriverNumber,Abbreviation,Position,Q1,Q2,Q3,Year,TeamName_Alfa Romeo Racing,TeamName_AlphaTauri,TeamName_Alpine,...,TeamName_Mercedes,TeamName_RB,TeamName_Racing Bulls,TeamName_Red Bull Racing,TeamName_Williams,BestQualiTime,Q2_Q1_Improvement,Q3_Q1_Improvement,RacePosition,Winner
0,33,VER,1.0,118.717,116.559,119.765,2021,False,False,False,...,False,False,False,True,False,116.559,2.158,-3.206,1.0,1
1,63,RUS,2.0,119.864,116.950,120.086,2021,False,False,False,...,False,False,False,False,True,116.950,2.914,-3.136,2.0,0
2,44,HAM,3.0,119.218,116.229,120.099,2021,False,False,False,...,True,False,False,False,False,116.229,2.989,-3.870,3.0,0
3,3,RIC,4.0,121.583,117.127,120.864,2021,False,False,False,...,False,False,False,False,False,117.127,4.456,-3.737,4.0,0
4,5,VET,5.0,120.175,116.814,120.935,2021,False,False,False,...,False,False,False,False,False,116.814,3.361,-4.121,5.0,0


In [29]:
model_df = merged_df.drop(columns=['DriverNumber','Abbreviation', 'RacePosition'])
bool_cols = model_df.select_dtypes(include='bool').columns
model_df[bool_cols] = model_df[bool_cols].astype(int)
model_df.head()

,Position,Q1,Q2,Q3,Year,TeamName_Alfa Romeo Racing,TeamName_AlphaTauri,TeamName_Alpine,TeamName_Aston Martin,TeamName_Audi,...,TeamName_McLaren,TeamName_Mercedes,TeamName_RB,TeamName_Racing Bulls,TeamName_Red Bull Racing,TeamName_Williams,BestQualiTime,Q2_Q1_Improvement,Q3_Q1_Improvement,Winner
0,1.0,118.717,116.559,119.765,2021,0,0,0,0,0,...,0,0,0,0,1,0,116.559,2.158,-3.206,1
1,2.0,119.864,116.950,120.086,2021,0,0,0,0,0,...,0,0,0,0,0,1,116.950,2.914,-3.136,0
2,3.0,119.218,116.229,120.099,2021,0,0,0,0,0,...,0,1,0,0,0,0,116.229,2.989,-3.870,0
3,4.0,121.583,117.127,120.864,2021,0,0,0,0,0,...,1,0,0,0,0,0,117.127,4.456,-3.737,0
4,5.0,120.175,116.814,120.935,2021,0,0,0,1,0,...,0,0,0,0,0,0,116.814,3.361,-4.121,0


In [30]:
print(model_df.dtypes)

Position                      float64
Q1                            float64
Q2                            float64
Q3                            float64
Year                            int64
TeamName_Alfa Romeo Racing      int64
TeamName_AlphaTauri             int64
TeamName_Alpine                 int64
TeamName_Aston Martin           int64
TeamName_Audi                   int64
TeamName_Cadillac               int64
TeamName_Ferrari                int64
TeamName_Haas F1 Team           int64
TeamName_Kick Sauber            int64
TeamName_McLaren                int64
TeamName_Mercedes               int64
TeamName_RB                     int64
TeamName_Racing Bulls           int64
TeamName_Red Bull Racing        int64
TeamName_Williams               int64
BestQualiTime                 float64
Q2_Q1_Improvement             float64
Q3_Q1_Improvement             float64
Winner                          int64
dtype: object


In [31]:
X = model_df.drop(columns=['Winner'])
y = model_df['Winner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(97, 23)
(97,)
(25, 23)
(25,)


In [32]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [33]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

In [35]:
print(classification_report(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96        23
           1       0.00      0.00      0.00         2

    accuracy                           0.92        25
   macro avg       0.46      0.50      0.48        25
weighted avg       0.85      0.92      0.88        25

ROC AUC: 0.9565217391304348


### Results prediction

In [38]:
X_full_scaled = scaler.transform(X)
probs = model.predict_proba(X_full_scaled)[:, 1]


results_df = model_df.copy()
results_df['WinProb'] = probs

results_df['Abbreviation'] = merged_df['Abbreviation'].values
results_df['Year'] = merged_df['Year'].values

In [39]:
X_full_scaled = scaler.transform(X)
probs = model.predict_proba(X_full_scaled)[:, 1]

results_df = pd.DataFrame({
    'Abbreviation': merged_df['Abbreviation'].values,
    'Year': merged_df['Year'].values,
    'TeamName': merged_df['TeamName'].values if 'TeamName' in merged_df.columns else None,
    'WinProb': probs,
    'ActualWinner': y.values
})

In [43]:
results_df['PredictedRank'] = results_df.groupby('Year')['WinProb'].rank(ascending=False, method='first')
predicted_podium = results_df[results_df['PredictedRank'] <= 3].sort_values(['Year', 'PredictedRank'])

# print(predicted_podium[['Year', 'Abbreviation', 'WinProb', 'PredictedRank']])
podium_df = pd.DataFrame(predicted_podium[['Year','Abbreviation', 'WinProb', 'PredictedRank']])
podium_df.head(20)

,Year,Abbreviation,WinProb,PredictedRank
0,2021,VER,0.121601,1.0
2,2021,HAM,0.121085,2.0
3,2021,RIC,0.073955,3.0
20,2022,VER,0.214550,1.0
22,2022,PER,0.160762,2.0
26,2022,HAM,0.124692,3.0
40,2023,VER,0.088360,1.0
43,2023,HAM,0.079081,2.0
42,2023,PER,0.065368,3.0
60,2024,VER,0.104133,1.0


This dataframe shows predicted podium for years 2021-2026. Checking with actual results

2021

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | VER       | VER    |
| 2nd   | HAM       | RUS    |
| 3rd   | RIC       | HAM    |

2022

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | VER       | VER    |
| 2nd   | PER       | PER    |
| 3rd   | HAM       | SAI    |

2023

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | VER       | VER    |
| 2nd   | HAM       | PER    |
| 3rd   | PER       | LEC    |

2024

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | VER       | HAM    |
| 2nd   | HAM       | PIA    |
| 3rd   | PER       | LEC    |

2025

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | NOR       | PIA    |
| 2nd   | PIA       | NOR    |
| 3rd   | RUS       | LEC    |

2026

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | ANT       | ANT    |
| 2nd   | RUS       | LEC    |
| 3rd   | VER       | VER    |


In [ ]:
%%sql


In [41]:
year_2026 = results_df[results_df['Year'] == 2026].copy()
year_2026['PredictedRank'] = year_2026['WinProb'].rank(ascending=False, method='first')

predicted_podium_2026 = year_2026[year_2026['PredictedRank'] <= 3].sort_values('PredictedRank')

print(predicted_podium_2026[['Abbreviation', 'WinProb', 'PredictedRank']])

    Abbreviation   WinProb  PredictedRank
100          ANT  0.172178            1.0
103          RUS  0.116945            2.0
101          VER  0.115669            3.0


### 2026 Winner prediction

| place | predicted | actual |
|-------|-----------|--------|
| 1st   | ANT       | ANT    |
| 2nd   | RUS       | LEC    |
| 3rd   | VER       | VER    |
